In [226]:
import os
import requests
import json
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from lxml import html
import re
from io import StringIO
import unicodedata

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

In [192]:


html_list = []

# Use version 144 to match your current Chrome version
driver = uc.Chrome(version_main=144)

try:
    # 1. Enter via the "Front Door"
    driver.get("https://fbref.com/")
    print("On homepage...")
    
    # Initialize the waiter
    wait = WebDriverWait(driver, 15)

    # 2
    # We wait for it to be present first
    comps_link = wait.until(EC.presence_of_element_located((By.LINK_TEXT, "Competitions")))
    
    print("Clicking Competitions...")
    driver.execute_script("arguments[0].scrollIntoView();", comps_link)
    time.sleep(3) # Small pause to look human
    driver.execute_script("arguments[0].click();", comps_link)
    
    # 3
    # This page can be long, so JS click is much safer here
    print("Looking for Premier League link...")
    pl_link = wait.until(EC.presence_of_element_located((By.LINK_TEXT, "Premier League")))
    
    driver.execute_script("arguments[0].scrollIntoView();", pl_link)
    time.sleep(7)
    driver.execute_script("arguments[0].click();", pl_link)

    # 4
    # Wait a moment for the URL to change
    print("Looking for Premier League Stats...")
    season = 2 # 2025->1, 2024->2, ...
    pl_link = wait.until(EC.presence_of_element_located((By.XPATH, f'//*[@id="seasons"]/tbody/tr[{season}]/td[1]/a')))
    
    driver.execute_script("arguments[0].scrollIntoView();", pl_link)
    time.sleep(3)
    driver.execute_script("arguments[0].click();", pl_link)

    pl_link = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="inner_nav"]/ul/li[2]/div/ul[1]/li[1]/a')))
    
    driver.execute_script("arguments[0].scrollIntoView();", pl_link)
    time.sleep(2)
    driver.execute_script("arguments[0].click();", pl_link)

    time.sleep(5)
    print(f"Successfully arrived! Current URL: {driver.current_url}")
    
    # Code for copying the HTML so that it could be processed using BeautifulSoup
    html = driver.page_source
    html_list.append(html)
except Exception as e:
    print(f"An error occurred: {e}")

finally:
    # Keeps the window open for 5 seconds so you can see the result
    time.sleep(5)
    driver.quit()

On homepage...
Clicking Competitions...
Looking for Premier League link...
Looking for Premier League Stats...
Successfully arrived! Current URL: https://fbref.com/en/comps/9/2024-2025/stats/2024-2025-Premier-League-Stats


In [198]:
soup = BeautifulSoup(html_list[0], 'html')

In [200]:
table = soup.find('table', id='stats_standard')

if table:
    stat_2024_df = pd.read_html(StringIO(str(table)))[0]
    
    stat_2024_df.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in df.columns.values]
    stat_2024_df.to_csv('fbref_pl_2025.csv')
    
    print("Table successfully created!")
    print(stat_2024_df.head())
else:
    print("Could not find the stats table.")

Table successfully created!
  Unnamed: 0_level_0_Rk Unnamed: 1_level_0_Player Unnamed: 2_level_0_Nation  \
0                     1                Max Aarons                   eng ENG   
1                     2         Joshua Acheampong                   eng ENG   
2                     3               Tyler Adams                    us USA   
3                     4          Tosin Adarabioyo                   eng ENG   
4                     5             Simon Adingra                    ci CIV   

  Unnamed: 3_level_0_Pos Unnamed: 4_level_0_Squad Unnamed: 5_level_0_Age  \
0                     DF              Bournemouth                     24   
1                     DF                  Chelsea                     18   
2                     MF              Bournemouth                     25   
3                     DF                  Chelsea                     26   
4                     MF                 Brighton                     22   

  Unnamed: 6_level_0_Born Playing Time_M

In [202]:
stat_2024_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 596 entries, 0 to 595
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Unnamed: 0_level_0_Rk        596 non-null    str  
 1   Unnamed: 1_level_0_Player    596 non-null    str  
 2   Unnamed: 2_level_0_Nation    595 non-null    str  
 3   Unnamed: 3_level_0_Pos       596 non-null    str  
 4   Unnamed: 4_level_0_Squad     596 non-null    str  
 5   Unnamed: 5_level_0_Age       595 non-null    str  
 6   Unnamed: 6_level_0_Born      595 non-null    str  
 7   Playing Time_MP              596 non-null    str  
 8   Playing Time_Starts          596 non-null    str  
 9   Playing Time_Min             596 non-null    str  
 10  Playing Time_90s             596 non-null    str  
 11  Performance_Gls              596 non-null    str  
 12  Performance_Ast              596 non-null    str  
 13  Performance_G+A              596 non-null    str  
 14  Perfo

In [206]:
pd.set_option('display.max_columns', None)

In [82]:
print("Lucá Williams Barnett" in df["Unnamed: 1_level_0_Player"].values.tolist())

False


going into: https://www.transfermarkt.us/premier-league/startseite/wettbewerb/GB1
going into: https://www.transfermarkt.us/laliga/startseite/wettbewerb/ES1
going into: https://www.transfermarkt.us/serie-a/startseite/wettbewerb/IT1
going into: https://www.transfermarkt.us/bundesliga/startseite/wettbewerb/L1
going into: https://www.transfermarkt.us/ligue-1/startseite/wettbewerb/FR1


In [166]:
team_2024_url_list[0:6]

['https://www.transfermarkt.us/manchester-city/startseite/verein/281/saison_id/2024',
 'https://www.transfermarkt.us/fc-arsenal/startseite/verein/11/saison_id/2024',
 'https://www.transfermarkt.us/fc-chelsea/startseite/verein/631/saison_id/2024',
 'https://www.transfermarkt.us/fc-liverpool/startseite/verein/31/saison_id/2024',
 'https://www.transfermarkt.us/tottenham-hotspur/startseite/verein/148/saison_id/2024',
 'https://www.transfermarkt.us/manchester-united/startseite/verein/985/saison_id/2024']

going into: https://www.transfermarkt.us/manchester-city/startseite/verein/281/saison_id/2024
going into: https://www.transfermarkt.us/fc-arsenal/startseite/verein/11/saison_id/2024
going into: https://www.transfermarkt.us/fc-chelsea/startseite/verein/631/saison_id/2024
going into: https://www.transfermarkt.us/fc-liverpool/startseite/verein/31/saison_id/2024
going into: https://www.transfermarkt.us/tottenham-hotspur/startseite/verein/148/saison_id/2024
going into: https://www.transfermarkt.us/manchester-united/startseite/verein/985/saison_id/2024
going into: https://www.transfermarkt.us/newcastle-united/startseite/verein/762/saison_id/2024
going into: https://www.transfermarkt.us/nottingham-forest/startseite/verein/703/saison_id/2024
going into: https://www.transfermarkt.us/crystal-palace/startseite/verein/873/saison_id/2024
going into: https://www.transfermarkt.us/aston-villa/startseite/verein/405/saison_id/2024
going into: https://www.transfermarkt.us/brighton-amp-hove-albion/startse

In [188]:
pd.DataFrame(list(season2024_name_to_value.items()), columns=['Name', 'Value (€)']).to_csv("name_value_2024.csv")

In [194]:
name_value_2024_df = pd.DataFrame(list(season2024_name_to_value.items()), columns=['Name', 'Value (€)'])

In [210]:
stat_2024_df = stat_2024_df.rename(columns={'Unnamed: 1_level_0_Player': 'Name'})

In [216]:
merged_2024 = normalize_and_merge(stats_df=stat_2024_df, name_value_df=name_value_2024_df)

In [218]:
merged_2024.head()

,Unnamed: 0_level_0_Rk,Name_x,Unnamed: 2_level_0_Nation,Unnamed: 3_level_0_Pos,Unnamed: 4_level_0_Squad,Unnamed: 5_level_0_Age,Unnamed: 6_level_0_Born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,Unnamed: 24_level_0_Matches,Name_Clean,Name_y,Value (€)
0,1,Max Aarons,eng ENG,DF,Bournemouth,24,2000,3,1,86,1.0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,Matches,max aarons,Max Aarons,6000000.0
1,2,Joshua Acheampong,eng ENG,DF,Chelsea,18,2006,4,2,170,1.9,0,0,0,0,0,0,1,0,0.00,0.00,0.00,0.00,0.00,Matches,joshua acheampong,NaN,NaN
2,3,Tyler Adams,us USA,MF,Bournemouth,25,1999,28,21,1965,21.8,0,3,3,0,0,0,7,0,0.00,0.14,0.14,0.00,0.14,Matches,tyler adams,Tyler Adams,18000000.0
3,4,Tosin Adarabioyo,eng ENG,DF,Chelsea,26,1997,22,15,1409,15.7,1,1,2,1,0,0,4,0,0.06,0.06,0.13,0.06,0.13,Matches,tosin adarabioyo,Tosin Adarabioyo,20000000.0
4,5,Simon Adingra,ci CIV,MF,Brighton,22,2002,29,12,1097,12.2,2,2,4,2,0,0,0,0,0.16,0.16,0.33,0.16,0.33,Matches,simon adingra,Simon Adingra,28000000.0


In [222]:
merged_2024.info()

<class 'pandas.DataFrame'>
RangeIndex: 597 entries, 0 to 596
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0_level_0_Rk        597 non-null    str    
 1   Name_x                       597 non-null    str    
 2   Unnamed: 2_level_0_Nation    596 non-null    str    
 3   Unnamed: 3_level_0_Pos       597 non-null    str    
 4   Unnamed: 4_level_0_Squad     597 non-null    str    
 5   Unnamed: 5_level_0_Age       596 non-null    str    
 6   Unnamed: 6_level_0_Born      596 non-null    str    
 7   Playing Time_MP              597 non-null    str    
 8   Playing Time_Starts          597 non-null    str    
 9   Playing Time_Min             597 non-null    str    
 10  Playing Time_90s             597 non-null    str    
 11  Performance_Gls              597 non-null    str    
 12  Performance_Ast              597 non-null    str    
 13  Performance_G+A              59

In [224]:
len(merged_2024)

597

In [297]:
def click_link(driver, link, wait_sec=3):
    driver.execute_script("arguments[0].scrollIntoView();", link)
    time.sleep(wait_sec)
    driver.execute_script("arguments[0].click();", link)
    print(f"Current URL: {driver.current_url}")

def stat_get_html(driver, wait, league, season):
    index = 2026 - season
    clicks = 0
    
    link = wait.until(EC.presence_of_element_located((By.LINK_TEXT, league)))
    click_link(driver, link, wait_sec=7)
    clicks += 1
    
    link = wait.until(EC.presence_of_element_located((By.XPATH, f'//*[@id="seasons"]/tbody/tr[{index}]/td[1]/a')))
    click_link(driver, link, wait_sec=3)
    clicks += 1

    link = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="inner_nav"]/ul/li[2]/div/ul[1]/li[1]/a')))
    click_link(driver, link, wait_sec=2)
    clicks += 1

    time.sleep(5)
    print(f"Successfully arrived! Current URL: {driver.current_url}")
    html = driver.page_source

    for _ in range(clicks):
        driver.back()
        print(f"Returned to: {driver.current_url}")
        time.sleep(1)

    return html

# season: 2025->1, 2024->2, ...
def get_stats_df(season):
    league_list = ["Premier League", "La Liga", "Ligue 1", "Fußball-Bundesliga", "Serie A"]
    final_df = None
    html_list = []
    
    driver = uc.Chrome(version_main=144)

    try:
        for league in league_list:
            driver.get("https://fbref.com/")

            wait = WebDriverWait(driver, 15)
            
            comps_link = wait.until(EC.presence_of_element_located((By.LINK_TEXT, "Competitions")))
            click_link(driver, comps_link, wait_sec=2)
            
            html = stat_get_html(driver, wait, league, season)
            html_list.append(html)
            print(f"Added html to the list, currently have {len(html_list)} htmls")

    except Exception as e:
        print(f"An error occurred: {e}")
    
    finally:
        time.sleep(5)
        driver.quit()

    if html_list:
        df_list = []
        for html in html_list:
            soup = BeautifulSoup(html, 'html')
            table = soup.find('table', id='stats_standard')
            
            if table:
                df = pd.read_html(StringIO(str(table)))[0]
                
                df.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in df.columns.values]
                df_list.append(df)
            else:
                print("Could not find the stats table.")
        final_df = pd.concat(df_list, ignore_index=True).rename(columns={'Unnamed: 1_level_0_Player': 'Name'})
        final_df.to_csv(f"stats_{season}.csv")
    else:
        print("html not found")

    return final_df
    

In [271]:
stat_for_2024 = get_pl_stats(2024)

Current URL: https://fbref.com/en/
Current URL: https://fbref.com/en/comps/
Current URL: https://fbref.com/en/comps/9/history/Premier-League-Seasons
Current URL: https://fbref.com/en/comps/9/2024-2025/2024-2025-Premier-League-Stats
Successfully arrived! Current URL: https://fbref.com/en/comps/9/2024-2025/stats/2024-2025-Premier-League-Stats
Returned to: https://fbref.com/en/comps/9/2024-2025/2024-2025-Premier-League-Stats
Returned to: https://fbref.com/en/comps/9/history/Premier-League-Seasons
Returned to: https://fbref.com/en/comps/
Added html to the list, currently have 1 htmls
Current URL: https://fbref.com/en/
Current URL: https://fbref.com/en/comps/
Current URL: https://fbref.com/en/comps/12/history/La-Liga-Seasons
Current URL: https://fbref.com/en/comps/12/2024-2025/2024-2025-La-Liga-Stats
Successfully arrived! Current URL: https://fbref.com/en/comps/12/2024-2025/stats/2024-2025-La-Liga-Stats
Returned to: https://fbref.com/en/comps/12/2024-2025/2024-2025-La-Liga-Stats
Returned to

In [313]:
def get_response(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36'
    }
    response = requests.get(url, headers=headers)

    return response

def create_url(team_name, team_id, season):
    return f"https://www.transfermarkt.us/{team_name}/startseite/verein/{team_id}/saison_id/{season}"

def parse_value(text):
    num_part = re.search(r'[0-9.]+', text).group()
    value = float(num_part)

    if 'm' in text:
        return value * 1_000_000
    elif 'k' in text:
        return value * 1_000
    elif 'bn' in text:
        return value * 1_000_000_000
    return value

def get_values_df(season):
    league_url_list = [
        "https://www.transfermarkt.us/premier-league/startseite/wettbewerb/GB1", 
        "https://www.transfermarkt.us/laliga/startseite/wettbewerb/ES1", 
        "https://www.transfermarkt.us/serie-a/startseite/wettbewerb/IT1", 
        "https://www.transfermarkt.us/bundesliga/startseite/wettbewerb/L1", 
        "https://www.transfermarkt.us/ligue-1/startseite/wettbewerb/FR1", 
    ]
    url_list = []
    
    team_url_list = []
    for league_url in league_url_list:
        print(f"going into: {league_url}")
        response = get_response(league_url)
        time.sleep(3)
    
        if response.status_code == 200:
            tree = html.fromstring(response.content)
            rows = tree.xpath('//*[@id="yw1"]/table/tbody/tr')
            for row in rows:
                links = row.xpath('./td[2]/a/@href')
                if links:
                    full_url = "https://www.transfermarkt.us" + links[0] + ""
                    team_url_list.append(full_url)

    print(f"Found {len(team_url_list)} Teams.")
    
    team_name_id_map = {}
    
    for team_url in team_url_list:
        team_url_elements = team_url.split("/")
        team_name = team_url_elements[team_url_elements.index("www.transfermarkt.us") + 1]
        team_id = team_url_elements[team_url_elements.index("verein") + 1]
        team_name_id_map[team_name] = team_id

    for team_name in team_name_id_map:
        url_list.append(create_url(team_name, team_name_id_map[team_name], season))

    name_to_value = {}

    num = 0
    length = len(url_list)
    for url in url_list:
        # print(f"going into: {url}")
        response = get_response(url)
        time.sleep(3)
    
        if response.status_code == 200:
            tree = html.fromstring(response.content)
            rows = tree.xpath('//*[@id="yw1"]/table/tbody/tr')
            for row in rows:
                name_list = row.xpath('.//td[2]//table//tr[1]//td[2]//a/text()')
                value_list = row.xpath('./td[6]/a/text()')
                if name_list and value_list:
                    name_text = name_list[0].strip()
                    value = parse_value(value_list[0].strip())
                    name_to_value[name_text] = value
        num += 1
        print(f"{num}/{length} Teams Added")
    final_df = pd.DataFrame(list(name_to_value.items()), columns=['Name', 'Value (€)'])
    final_df.to_csv(f"name_value_{season}.csv")
    return final_df

In [311]:
def normalize_name(name):
    if not isinstance(name, str): return ""
    name = unicodedata.normalize('NFD', name).encode('ascii', 'ignore').decode("utf-8")
    return name.lower().strip()

def normalize_and_merge(stats_df, name_value_df):
    stats_df['Name_Clean'] = stats_df['Name'].apply(normalize_name)
    name_value_df['Name_Clean'] = name_value_df['Name'].apply(normalize_name)

    return pd.merge(stats_df, name_value_df, on='Name_Clean', how='left')

def get_full_dataset(season):
    stats_df = get_stats_df(season)
    value_df = get_values_df(season)
    merged_df = normalize_and_merge(stats_df=stats_df, name_value_df=value_df)
    merged_df.to_csv(f"{season}.csv")

    return merged_df

In [309]:
value_2023_df = get_values_df(2023)

going into: https://www.transfermarkt.us/premier-league/startseite/wettbewerb/GB1
going into: https://www.transfermarkt.us/laliga/startseite/wettbewerb/ES1
going into: https://www.transfermarkt.us/serie-a/startseite/wettbewerb/IT1
going into: https://www.transfermarkt.us/bundesliga/startseite/wettbewerb/L1
going into: https://www.transfermarkt.us/ligue-1/startseite/wettbewerb/FR1
going into: https://www.transfermarkt.us/manchester-city/startseite/verein/281/saison_id/2023
going into: https://www.transfermarkt.us/fc-arsenal/startseite/verein/11/saison_id/2023
going into: https://www.transfermarkt.us/fc-chelsea/startseite/verein/631/saison_id/2023
going into: https://www.transfermarkt.us/fc-liverpool/startseite/verein/31/saison_id/2023
going into: https://www.transfermarkt.us/tottenham-hotspur/startseite/verein/148/saison_id/2023
going into: https://www.transfermarkt.us/manchester-united/startseite/verein/985/saison_id/2023
going into: https://www.transfermarkt.us/newcastle-united/starts

In [312]:
value_2023_df.head()

,Name,Value (€)
0,Ederson,35000000.0
1,Stefan Ortega,9000000.0
2,Zack Steffen,2000000.0
3,True Grant,500000.0
4,Scott Carson,200000.0
